# EXP_009d3: Random Baseline — ATR Null Hypothesis

## Depends On
**Stage 1 (Attractor Dominance) must be complete before running this notebook.**

## Scientific Purpose

This experiment constitutes the **statistical null hypothesis** for the Activation Tensor Resonance method.

Stage 1 demonstrated that 125 real prompts converge to 5 attractor basins. But a critical question remains:

> *Does the ATR loop converge to the same basins regardless of input, or only for inputs that lie on the training data manifold?*

### Design

We replace the prompt-derived residual stream tensor with **random Gaussian noise** of matched:
- **Shape:** `[seq_len, d_model]` — sampled from the distribution of real prompt sequence lengths
- **Norm:** Scaled to match the mean Frobenius norm of real Stage 1 initial tensors
- **Count:** 125 trials (matching Stage 1 exactly)

Everything else is **identical** to Stage 1: same model, same layer range (0→11), same iteration schedule, same L2 normalisation.

### Three Possible Outcomes

| Outcome | Interpretation |
|:---|:---|
| **Same 5 basins** | Basins are intrinsic spectral properties of the weight geometry (pure eigenvoice) |
| **Different basins** | The 5 basins are specific to the manifold region that real text occupies |
| **No convergence** | ATR requires structured (on-manifold) input to produce stable attractors |

**All three outcomes are publishable.** This experiment contextualises the Stage 1 findings.

---

In [ ]:
# ============================================================
# STEP 0: DEPENDENCIES
# ============================================================
import sys
!{sys.executable} -m pip install kaleido -q

In [ ]:
# ============================================================
# STEP 1: CALIBRATION
# ============================================================
import torch
import numpy as np
import os
import json
from collections import Counter
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Architecture: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")
print(f"Running on: {device}")

# Output directory
OUTPUT_DIR = "output_random_baseline"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

In [ ]:
# ============================================================
# STEP 2: CONFIGURATION — Matched to Stage 1
# ============================================================

N_TRIALS = 125  # Match Stage 1 prompt count exactly
RANDOM_SEED = 42  # Fixed seed for reproducibility

# Identical schedule to Stage 1
ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

LAYER_START = 0
LAYER_END = model.cfg.n_layers - 1  # 11 for GPT-2 Small
D_MODEL = model.cfg.d_model  # 768

# --- Calibrate from Stage 1 data ---
# Load actual Stage 1 results to extract real tensor statistics
stage1_path = os.path.join("output", "stage1_results.pt")
if os.path.exists(stage1_path):
    stage1_data = torch.load(stage1_path, map_location='cpu', weights_only=False)
    real_norms = []
    real_seq_lens = []
    for prompt_id, snapshots in stage1_data.items():
        # Iteration 0 snapshot contains the initial tensor
        iter0 = [s for s in snapshots if s['iteration'] == 0]
        if iter0:
            tensor = iter0[0]['tensor']  # shape: [seq_len, d_model]
            real_norms.append(tensor.norm().item())
            real_seq_lens.append(tensor.shape[0])
    
    MEAN_NORM = np.mean(real_norms)
    STD_NORM = np.std(real_norms)
    MEAN_SEQ_LEN = int(np.mean(real_seq_lens))
    MIN_SEQ_LEN = min(real_seq_lens)
    MAX_SEQ_LEN = max(real_seq_lens)
    
    print(f"Calibrated from {len(real_norms)} Stage 1 tensors:")
    print(f"  Frobenius norm: {MEAN_NORM:.2f} ± {STD_NORM:.2f}")
    print(f"  Sequence lengths: {MIN_SEQ_LEN}–{MAX_SEQ_LEN} (mean={MEAN_SEQ_LEN})")
else:
    # Fallback: typical values for GPT-2 Small with ~10 token prompts
    MEAN_NORM = 120.0  # Approximate
    STD_NORM = 20.0
    MEAN_SEQ_LEN = 10
    MIN_SEQ_LEN = 2
    MAX_SEQ_LEN = 16
    print("⚠ Stage 1 data not found — using fallback norm estimates")

print(f"\nExperimental config:")
print(f"  Trials: {N_TRIALS}")
print(f"  Seed: {RANDOM_SEED}")
print(f"  Schedule: {ITERATION_SCHEDULE}")
print(f"  Layers: {LAYER_START} → {LAYER_END}")

# Save config
config = {
    "experiment": "EXP_009d3_random_baseline",
    "n_trials": N_TRIALS,
    "seed": RANDOM_SEED,
    "schedule": ITERATION_SCHEDULE,
    "layers": f"{LAYER_START}->{LAYER_END}",
    "d_model": D_MODEL,
    "calibration": {
        "mean_norm": float(MEAN_NORM),
        "std_norm": float(STD_NORM),
        "mean_seq_len": MEAN_SEQ_LEN,
        "min_seq_len": MIN_SEQ_LEN,
        "max_seq_len": MAX_SEQ_LEN,
    },
    "device": device,
}
with open(os.path.join(OUTPUT_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)
print(f"\n[SAVED] {OUTPUT_DIR}/config.json")

In [ ]:
# ============================================================
# STEP 3: ENGINE — Identical ATR core, modified for tensor input
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions.
    Applies the Final LayerNorm before unembedding for correct decoding."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_atr_from_tensor(model, initial_tensor, layer_start, layer_end, max_iter, schedule,
                        dummy_prompt="The"):
    """
    ATR loop starting from an ARBITRARY tensor (not derived from a prompt).
    
    Uses a dummy prompt ("The") solely to provide the tokeniser with a valid
    input sequence. The dummy prompt's residual stream is immediately overwritten
    by the injection hook at layer_start, so its content has zero influence on
    the output.
    
    This is the ONLY difference from run_total_resonance_loop in Stage 1.
    """
    snapshots = []
    hook_point_read  = f"blocks.{layer_end}.hook_resid_post"
    hook_point_write = f"blocks.{layer_start}.hook_resid_pre"
    
    seq_len = initial_tensor.shape[0]
    current_tensor = initial_tensor.clone().to(model.cfg.device)
    initial_norm = current_tensor.norm().item()
    
    # We need a dummy prompt whose tokenised length matches seq_len.
    # Pad with repeated tokens to get the right shape.
    dummy_tokens = model.to_tokens(dummy_prompt)  # shape: [1, n_tok]
    n_dummy = dummy_tokens.shape[1]
    if n_dummy < seq_len:
        # Repeat the last token to fill the sequence
        padding = dummy_tokens[0, -1:].repeat(1, seq_len - n_dummy)
        dummy_tokens = torch.cat([dummy_tokens, padding], dim=1)
    elif n_dummy > seq_len:
        dummy_tokens = dummy_tokens[:, :seq_len]
    
    # --- Iteration 0: decode the random tensor directly ---
    last_vec = current_tensor[-1, :].clone()
    mean_vec = current_tensor.mean(dim=0).clone()
    
    if 0 in schedule:
        top_tokens_last = get_top_tokens(model, last_vec)
        all_pos_tokens = []
        for pos in range(seq_len):
            pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
            all_pos_tokens.append(pos_top[0][0])
        snapshots.append({
            "iteration": 0,
            "tensor": current_tensor.clone().cpu(),
            "last_vector": last_vec.clone().cpu(),
            "mean_vector": mean_vec.clone().cpu(),
            "last_norm": last_vec.norm().item(),
            "mean_norm": mean_vec.norm().item(),
            "tensor_norm": current_tensor.norm().item(),
            "top_tokens": top_tokens_last,
            "all_position_tokens": all_pos_tokens,
            "cosine_sim_last": 1.0,
            "cosine_sim_mean": 1.0,
            "position_similarity": 0.0,  # Random vectors should be near-orthogonal
        })
    
    prev_last = last_vec.clone()
    prev_mean = mean_vec.clone()
    
    # --- Main ATR loop ---
    for i in range(1, max_iter + 1):
        # L2 normalise to maintain energy level (identical to Stage 1)
        current_norm = current_tensor.norm().item()
        if current_norm > 0:
            current_tensor = current_tensor * (initial_norm / current_norm)
        
        inject_tensor = current_tensor.clone()
        
        def injection_hook(resid, hook, tensor=inject_tensor):
            resid[0, :, :] = tensor
            return resid
        
        model.add_hook(hook_point_write, injection_hook)
        try:
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    dummy_tokens,
                    names_filter=lambda n: n == hook_point_read
                )
        finally:
            model.reset_hooks()
        
        current_tensor = cache[hook_point_read][0].clone()
        last_vec = current_tensor[-1, :].clone()
        mean_vec = current_tensor.mean(dim=0).clone()
        
        if i in schedule:
            cos_sim_last = torch.nn.functional.cosine_similarity(
                last_vec.unsqueeze(0), prev_last.unsqueeze(0)
            ).item()
            cos_sim_mean = torch.nn.functional.cosine_similarity(
                mean_vec.unsqueeze(0), prev_mean.unsqueeze(0)
            ).item()
            
            pos_norms = current_tensor.norm(dim=1, keepdim=True).clamp(min=1e-8)
            normalized_positions = current_tensor / pos_norms
            pos_sim_matrix = normalized_positions @ normalized_positions.T
            mask = ~torch.eye(seq_len, dtype=torch.bool, device=pos_sim_matrix.device)
            position_similarity = pos_sim_matrix[mask].mean().item()
            
            top_tokens_last = get_top_tokens(model, last_vec)
            all_pos_tokens = []
            for pos in range(seq_len):
                pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
                all_pos_tokens.append(pos_top[0][0])
            
            snapshots.append({
                "iteration": i,
                "tensor": current_tensor.clone().cpu(),
                "last_vector": last_vec.clone().cpu(),
                "mean_vector": mean_vec.clone().cpu(),
                "last_norm": last_vec.norm().item(),
                "mean_norm": mean_vec.norm().item(),
                "tensor_norm": current_tensor.norm().item(),
                "top_tokens": top_tokens_last,
                "all_position_tokens": all_pos_tokens,
                "cosine_sim_last": cos_sim_last,
                "cosine_sim_mean": cos_sim_mean,
                "position_similarity": position_similarity,
            })
            print(f"  iter {i:>3}: top='{top_tokens_last[0][0].strip()}', "
                  f"cos_mean={cos_sim_mean:.4f}, pos_collapse={position_similarity:.4f}")
        
        prev_last = last_vec.clone()
        prev_mean = mean_vec.clone()
    
    return snapshots

print("Engine loaded.")

In [ ]:
# ============================================================
# STEP 4: GENERATE RANDOM TENSORS & RUN ATR
# ============================================================

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Pre-generate sequence lengths — sample from the real distribution
if 'real_seq_lens' in dir() and len(real_seq_lens) > 0:
    # Sample with replacement from actual Stage 1 sequence lengths
    seq_lengths = [real_seq_lens[i % len(real_seq_lens)] for i in range(N_TRIALS)]
else:
    seq_lengths = [MEAN_SEQ_LEN] * N_TRIALS

all_results = {}

for trial_idx in range(N_TRIALS):
    trial_id = f"R{trial_idx+1:03d}"
    seq_len = seq_lengths[trial_idx]
    
    # Generate random Gaussian tensor
    random_tensor = torch.randn(seq_len, D_MODEL)
    
    # Scale to match real norm distribution
    # Sample a norm from the observed distribution for this trial
    target_norm = MEAN_NORM + (torch.randn(1).item() * STD_NORM)
    target_norm = max(target_norm, 10.0)  # floor to avoid degenerate cases
    current_norm = random_tensor.norm().item()
    if current_norm > 0:
        random_tensor = random_tensor * (target_norm / current_norm)
    
    print(f"\n{'='*60}")
    print(f"[{trial_idx+1}/{N_TRIALS}] TRIAL: '{trial_id}'")
    print(f"  Shape: [{seq_len}, {D_MODEL}], Norm: {random_tensor.norm().item():.2f}")
    print(f"{'='*60}")
    
    snapshots = run_atr_from_tensor(
        model, random_tensor,
        LAYER_START, LAYER_END,
        MAX_ITERATIONS, ITERATION_SCHEDULE
    )
    
    terminal_token = snapshots[-1]['top_tokens'][0][0].strip()
    print(f"  ✓ Terminal token: '{terminal_token}'")
    
    all_results[trial_id] = snapshots

# Save raw results
torch.save(all_results, os.path.join(OUTPUT_DIR, 'random_baseline_results.pt'))
print(f"\n[SAVED] {OUTPUT_DIR}/random_baseline_results.pt")
print(f"Total trials: {len(all_results)}")

In [ ]:
# ============================================================
# STEP 5: ANALYSIS — Basin Comparison (Random vs. Real)
# ============================================================

# --- Extract terminal tokens from random baseline ---
random_terminals = {}
for trial_id, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    random_terminals[trial_id] = terminal

random_basin_counts = Counter(random_terminals.values())
random_n_basins = len(random_basin_counts)

# --- Load Stage 1 terminals for comparison ---
real_terminals = {}
if os.path.exists(stage1_path):
    for prompt_id, snapshots in stage1_data.items():
        terminal_snap = max(snapshots, key=lambda s: s['iteration'])
        terminal = terminal_snap['top_tokens'][0][0].strip()
        real_terminals[prompt_id] = terminal

real_basin_counts = Counter(real_terminals.values())
real_n_basins = len(real_basin_counts)

# --- Print comparison ---
print("="*60)
print("BASIN COMPARISON: Random Baseline vs. Real Prompts")
print("="*60)
print(f"\nReal prompts (Stage 1):  {real_n_basins} basins from {len(real_terminals)} prompts")
for token, count in real_basin_counts.most_common():
    pct = 100 * count / len(real_terminals)
    print(f"  '{token}': {count} ({pct:.1f}%)")

print(f"\nRandom baseline:  {random_n_basins} basins from {len(random_terminals)} trials")
for token, count in random_basin_counts.most_common(20):
    pct = 100 * count / len(random_terminals)
    print(f"  '{token}': {count} ({pct:.1f}%)")
if random_n_basins > 20:
    print(f"  ... and {random_n_basins - 20} more unique tokens")

# --- Overlap analysis ---
real_basin_set = set(real_basin_counts.keys())
random_basin_set = set(random_basin_counts.keys())
overlap = real_basin_set & random_basin_set

print(f"\nOverlap: {len(overlap)} shared basins: {overlap if overlap else '∅'}")
print(f"Real-only: {real_basin_set - random_basin_set}")
print(f"Random-only: {random_basin_set - real_basin_set}")

In [ ]:
# ============================================================
# STEP 6: CONVERGENCE DYNAMICS — Does random even converge?
# ============================================================

# Extract convergence metrics across iterations
convergence_data = {"trial": [], "iteration": [], "cos_sim_mean": [],
                    "pos_collapse": [], "top_token": []}

for trial_id, snapshots in all_results.items():
    for snap in snapshots:
        convergence_data["trial"].append(trial_id)
        convergence_data["iteration"].append(snap["iteration"])
        convergence_data["cos_sim_mean"].append(snap["cosine_sim_mean"])
        convergence_data["pos_collapse"].append(snap["position_similarity"])
        convergence_data["top_token"].append(snap["top_tokens"][0][0].strip())

# Compute per-iteration statistics
iter_stats = {}
for it in ITERATION_SCHEDULE:
    mask = [i for i, x in enumerate(convergence_data["iteration"]) if x == it]
    cos_vals = [convergence_data["cos_sim_mean"][i] for i in mask]
    pos_vals = [convergence_data["pos_collapse"][i] for i in mask]
    tokens_at_iter = [convergence_data["top_token"][i] for i in mask]
    n_unique = len(set(tokens_at_iter))
    iter_stats[it] = {
        "cos_mean": np.mean(cos_vals) if cos_vals else 0,
        "cos_std": np.std(cos_vals) if cos_vals else 0,
        "pos_mean": np.mean(pos_vals) if pos_vals else 0,
        "pos_std": np.std(pos_vals) if pos_vals else 0,
        "n_unique_tokens": n_unique,
    }

print("Convergence Dynamics (Random Baseline)")
print(f"{'Iter':>6} | {'cos_sim_mean':>14} | {'pos_collapse':>14} | {'n_unique':>10}")
print("-" * 55)
for it in ITERATION_SCHEDULE:
    s = iter_stats[it]
    print(f"{it:>6} | {s['cos_mean']:>7.4f}±{s['cos_std']:<6.4f} | "
          f"{s['pos_mean']:>7.4f}±{s['pos_std']:<6.4f} | {s['n_unique_tokens']:>10}")

In [ ]:
# ============================================================
# STEP 7: VISUALISATION — Side-by-Side Basin Distributions
# ============================================================

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Real Prompts (Stage 1): {real_n_basins} basins",
        f"Random Baseline: {random_n_basins} basins"
    ),
    horizontal_spacing=0.12
)

# Real basins
real_sorted = real_basin_counts.most_common()
fig.add_trace(
    go.Bar(
        x=[t[0] for t in real_sorted],
        y=[t[1] for t in real_sorted],
        marker_color='#636EFA',
        name='Real',
        text=[f"{100*t[1]/125:.0f}%" for t in real_sorted],
        textposition='outside'
    ),
    row=1, col=1
)

# Random basins (show top 15 to keep readable)
random_sorted = random_basin_counts.most_common(15)
fig.add_trace(
    go.Bar(
        x=[t[0] for t in random_sorted],
        y=[t[1] for t in random_sorted],
        marker_color='#EF553B',
        name='Random',
        text=[f"{100*t[1]/125:.0f}%" for t in random_sorted],
        textposition='outside'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="ATR Basin Distribution: Real Prompts vs. Random Baseline (GPT-2 Small)",
    height=500,
    showlegend=False,
    template='plotly_dark'
)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.write_image(os.path.join(OUTPUT_DIR, 'basin_comparison.png'), scale=2)
fig.show()
print(f"[SAVED] {OUTPUT_DIR}/basin_comparison.png")

In [ ]:
# ============================================================
# STEP 8: VISUALISATION — Convergence Rate Comparison
# ============================================================

# n_unique_tokens over iterations for random baseline
iters = [it for it in ITERATION_SCHEDULE if it > 0]
n_unique_random = [iter_stats[it]['n_unique_tokens'] for it in iters]

# Same metric for real prompts
n_unique_real = []
if stage1_data:
    for it in iters:
        tokens_at_iter = []
        for prompt_id, snapshots in stage1_data.items():
            snap = [s for s in snapshots if s['iteration'] == it]
            if snap:
                tokens_at_iter.append(snap[0]['top_tokens'][0][0].strip())
        n_unique_real.append(len(set(tokens_at_iter)))

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=iters, y=n_unique_real,
    mode='lines+markers',
    name='Real Prompts',
    line=dict(color='#636EFA', width=3),
    marker=dict(size=10)
))

fig2.add_trace(go.Scatter(
    x=iters, y=n_unique_random,
    mode='lines+markers',
    name='Random Baseline',
    line=dict(color='#EF553B', width=3, dash='dash'),
    marker=dict(size=10, symbol='diamond')
))

fig2.update_layout(
    title='Attractor Consolidation: Real vs. Random',
    xaxis_title='ATR Iteration',
    yaxis_title='Number of Unique Terminal Tokens',
    xaxis_type='log',
    template='plotly_dark',
    height=450,
    legend=dict(x=0.7, y=0.95)
)

fig2.write_image(os.path.join(OUTPUT_DIR, 'convergence_comparison.png'), scale=2)
fig2.show()
print(f"[SAVED] {OUTPUT_DIR}/convergence_comparison.png")

In [ ]:
# ============================================================
# STEP 9: VISUALISATION — Position Collapse Comparison
# ============================================================

pos_collapse_random = [iter_stats[it]['pos_mean'] for it in iters]

# Same metric for real prompts
pos_collapse_real = []
if stage1_data:
    for it in iters:
        vals = []
        for prompt_id, snapshots in stage1_data.items():
            snap = [s for s in snapshots if s['iteration'] == it]
            if snap:
                vals.append(snap[0]['position_similarity'])
        pos_collapse_real.append(np.mean(vals) if vals else 0)

fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=iters, y=pos_collapse_real,
    mode='lines+markers',
    name='Real Prompts',
    line=dict(color='#636EFA', width=3),
    marker=dict(size=10)
))

fig3.add_trace(go.Scatter(
    x=iters, y=pos_collapse_random,
    mode='lines+markers',
    name='Random Baseline',
    line=dict(color='#EF553B', width=3, dash='dash'),
    marker=dict(size=10, symbol='diamond')
))

fig3.add_hline(y=1.0, line_dash='dot', line_color='gray',
              annotation_text='Total collapse (all positions identical)')

fig3.update_layout(
    title='Position Collapse Rate: Real vs. Random',
    xaxis_title='ATR Iteration',
    yaxis_title='Mean Cross-Position Cosine Similarity',
    xaxis_type='log',
    yaxis_range=[-0.1, 1.1],
    template='plotly_dark',
    height=450,
    legend=dict(x=0.02, y=0.95)
)

fig3.write_image(os.path.join(OUTPUT_DIR, 'position_collapse_comparison.png'), scale=2)
fig3.show()
print(f"[SAVED] {OUTPUT_DIR}/position_collapse_comparison.png")

In [ ]:
# ============================================================
# STEP 10: DISSOLUTION PATHWAYS — Random Baseline
# ============================================================

# Show first 20 trials' dissolution pathways (matching Stage 1 format)
lines = ["# Dissolution Pathways — Random Baseline\n"]
lines.append(f"## {N_TRIALS} Random Gaussian Tensors (seed={RANDOM_SEED})\n")

# Header
trial_ids = sorted(all_results.keys())[:20]
header = "| Iter | " + " | ".join(trial_ids) + " |"
sep = "| :--- | " + " | ".join([":---"] * len(trial_ids)) + " |"
lines.append(header)
lines.append(sep)

for it in ITERATION_SCHEDULE:
    row = [f"**{it}**"]
    for tid in trial_ids:
        snap = [s for s in all_results[tid] if s['iteration'] == it]
        if snap:
            token = snap[0]['top_tokens'][0][0].strip()
            row.append(f"`{token}`")
        else:
            row.append("—")
    lines.append("| " + " | ".join(row) + " |")

pathway_md = "\n".join(lines)

with open(os.path.join(OUTPUT_DIR, 'dissolution_pathways_random.md'), 'w', encoding='utf-8') as f:
    f.write(pathway_md)

display(Markdown(pathway_md))
print(f"\n[SAVED] {OUTPUT_DIR}/dissolution_pathways_random.md")

In [ ]:
# ============================================================
# STEP 11: STATISTICAL TEST — Permutation test for basin count
# ============================================================

# Question: Is the real basin count (5) significantly different from
# the random baseline basin count?

# Bootstrap the random baseline to get a CI on its basin count
n_bootstrap = 10000
random_terminal_list = list(random_terminals.values())
bootstrap_basin_counts = []

np.random.seed(RANDOM_SEED)
for _ in range(n_bootstrap):
    sample = np.random.choice(random_terminal_list, size=len(random_terminal_list), replace=True)
    bootstrap_basin_counts.append(len(set(sample)))

bootstrap_mean = np.mean(bootstrap_basin_counts)
bootstrap_ci_low = np.percentile(bootstrap_basin_counts, 2.5)
bootstrap_ci_high = np.percentile(bootstrap_basin_counts, 97.5)

print("="*60)
print("STATISTICAL TEST: Basin Count Significance")
print("="*60)
print(f"\nReal prompts:    {real_n_basins} basins")
print(f"Random baseline: {random_n_basins} basins")
print(f"Bootstrap (random): {bootstrap_mean:.1f} basins (95% CI: [{bootstrap_ci_low:.0f}, {bootstrap_ci_high:.0f}])")

if real_n_basins < bootstrap_ci_low:
    print(f"\n✅ SIGNIFICANT: Real basin count ({real_n_basins}) is BELOW the random 95% CI.")
    print("   → The weight geometry funnels real text into FEWER attractors than random noise.")
    print("   → The 5-basin landscape is a property of the on-manifold region.")
elif real_n_basins > bootstrap_ci_high:
    print(f"\n✅ SIGNIFICANT: Real basin count ({real_n_basins}) is ABOVE the random 95% CI.")
    print("   → Real text explores MORE of the attractor landscape than random noise.")
else:
    print(f"\n⚠ NOT SIGNIFICANT: Real basin count ({real_n_basins}) is WITHIN the random 95% CI.")
    print("   → Cannot distinguish real from random on basin count alone.")
    print("   → Check basin IDENTITY overlap instead.")

In [ ]:
# ============================================================
# STEP 12: FINAL REPORT — Automated interpretation
# ============================================================

# Compute key metrics for the report
overlap_pct = 100 * len(overlap) / max(len(real_basin_set), 1)
random_converged = iter_stats[100]['pos_mean'] > 0.99  # Did positions collapse?
random_cos_converged = iter_stats[100]['cos_mean'] > 0.99

report = f"""# EXP_009d3: Random Baseline — Results Summary

## Experimental Parameters
- **Trials:** {N_TRIALS} random Gaussian tensors
- **Seed:** {RANDOM_SEED}
- **Norm calibration:** {MEAN_NORM:.2f} ± {STD_NORM:.2f} (from Stage 1)
- **Iteration schedule:** {ITERATION_SCHEDULE}
- **Model:** GPT-2 Small (12L, 768d)

## Key Results

| Metric | Real (Stage 1) | Random Baseline |
|:---|:---|:---|
| Terminal basins | {real_n_basins} | {random_n_basins} |
| Position collapse at iter 100 | ~1.000 | {iter_stats[100]['pos_mean']:.4f} |
| Cosine convergence at iter 100 | ~1.000 | {iter_stats[100]['cos_mean']:.4f} |
| Basin overlap | — | {len(overlap)}/{real_n_basins} ({overlap_pct:.0f}%) |

## Interpretation

### Does random input converge?
Position collapse: {'YES ✅' if random_converged else 'NO ❌'}
Cosine convergence: {'YES ✅' if random_cos_converged else 'NO ❌'}

### Basin identity overlap
Shared basins: {overlap if overlap else '∅ (none)'}
Real-only basins: {real_basin_set - random_basin_set}
Random-only basins: {random_basin_set - real_basin_set if len(random_basin_set - real_basin_set) < 20 else f'({len(random_basin_set - real_basin_set)} unique tokens)'}

### Bootstrap significance
Random basin count: {bootstrap_mean:.1f} (95% CI: [{bootstrap_ci_low:.0f}, {bootstrap_ci_high:.0f}])
Real basin count: {real_n_basins}

## Outcome Classification

Based on the results above, this experiment falls into one of three categories:

1. **Same basins** → Eigenvoice is intrinsic to weight geometry
2. **Different basins** → Eigenvoice is manifold-specific
3. **No convergence** → ATR requires structured input

---
*Generated automatically by EXP_009d3*
"""

with open(os.path.join(OUTPUT_DIR, 'random_baseline_report.md'), 'w', encoding='utf-8') as f:
    f.write(report)

display(Markdown(report))
print(f"\n[SAVED] {OUTPUT_DIR}/random_baseline_report.md")
print("\n" + "="*60)
print("EXP_009d3 COMPLETE")
print("="*60)